# InternVL3 reasoning-arm screen: 3 pre-registered ideas, one shot each

Thirteenth notebook. `pilot/12_internvl3_fermat.ipynb`'s reference run found
InternVL3-8B's `has_error=1` stratified reasoning AUROC at **0.628
[0.548, 0.706]** -- powered (45 misgraded items, clears the n=30 minimum)
and its CI excludes chance, but it does not clear this project's 0.70
confirmation threshold. Rather than iterating on fixes until something
clears 0.70 (rejected as a p-hacking risk -- the same failure class as the
K=15 reasoning-text cascade episode), this notebook runs a **small,
pre-registered, single-shot screen**: three specific ideas, each tied to a
diagnosed mechanism, tested once each, with all three results reported
honestly regardless of outcome.

**Screen population**: the same 150 `has_error=1` items from notebook 12's
reference sample (`pilot.data.load_fermat_balanced(n=300, seed=42,
target_error_frac=0.5)`, filtered to `has_error==1`) -- not a fresh draw,
so results are directly comparable to the 0.628 baseline on the exact same
items.

**The three ideas:**
1. `commit` -- an explicit "don't default to no-error on illegible input"
   instruction (`pilot.prompts.GRADING_USER_PROMPT_COMMIT`). Motivated by a
   real wrong-vote case whose own reasoning states the image is unreadable
   and defaults to "no error" as a result.
2. `restate` -- reuse the restate-result-first variant already screened on
   3B (`pilot.prompts.GRADING_USER_PROMPT_RESTATE`), unmodified. It didn't
   fix 3B's far more extreme yes-bias, but InternVL3 starts from a much
   milder bias (59.7% says-error vs 3B's 93%) -- worth one honest retest.
3. `k10` -- draw 5 more *baseline*-prompt grading samples per item and
   combine with the 5 already collected in notebook 12's reference run to
   reach K=10, mirroring the confirmed K=5->K=10 perception sharpening.
   Motivated directly by the numbers: the has_error=1 CI upper bound
   (0.706) sits just under 0.70, the profile a too-few-samples problem
   produces.

**Pre-registered pass bar** (written before this notebook runs, not after
seeing results): `has_error=1` AUROC >= 0.70 AND CI excludes chance AND
minority class (min of n_wrong, n_correct) >= 30, evaluated on this
150-item screen sample. **If none pass, 0.628 stands as the reported
InternVL3 reasoning result and no further InternVL3 GPU work is planned.**
If one passes, that is flagged for a separate decision (a full n=300+
confirmation run), not treated as an automatic next step.


In [1]:
# Install cell: GPU-dependent packages only. Identical to notebook 12.
%pip install -q transformers accelerate datasets huggingface_hub bitsandbytes


In [2]:
# Auth & code/results access cell. Identical to notebooks 06/09/10/11/12.
import json
import os
from getpass import getpass

from huggingface_hub import login

from google.colab import drive

drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/uncertainty-math-vlm"
DRIVE_MODEL_CACHE = f"{PROJECT_DIR}/model_cache"
os.makedirs(DRIVE_MODEL_CACHE, exist_ok=True)

TOKEN_FILE = f"{PROJECT_DIR}/.tokens.json"
RESET_TOKENS = False


def get_token(name, prompt):
    tokens = {}
    if os.path.exists(TOKEN_FILE):
        with open(TOKEN_FILE) as f:
            tokens = json.load(f)
    if RESET_TOKENS or not tokens.get(name):
        tokens[name] = getpass(prompt).strip()
        with open(TOKEN_FILE, "w") as f:
            json.dump(tokens, f)
        os.chmod(TOKEN_FILE, 0o600)
        print(f"Saved {name} to Drive -- you will not be asked for it again.")
    return tokens[name]


HF_TOKEN = get_token("HF_TOKEN", "Hugging Face token (asked once): ")
GH_TOKEN = get_token("GH_TOKEN", "GitHub token with 'repo' scope (asked once): ")

if not HF_TOKEN.startswith("hf_"):
    raise ValueError(
        "Stored Hugging Face token does not start with 'hf_'. Set "
        "RESET_TOKENS = True and re-run this cell to replace it."
    )

login(token=HF_TOKEN)
print("Hugging Face login OK")

REPO_URL = "https://github.com/sepehrmaleki369/uncertainty-math-vlm.git"
!rm -rf repo
!git clone -q {REPO_URL} repo
%pip install -q -e repo/

import importlib
import sys

REPO_DIR = os.path.abspath("repo")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
importlib.invalidate_caches()

import pilot.data
import pilot.prompts
import pilot.parsing
import pilot.entropy
import pilot.canonicalize
import pilot.plotting

print(f"pilot package imported from: {os.path.dirname(pilot.__file__)}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Hugging Face login OK
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for pilot (pyproject.toml) ... done
pilot package imported from: /content/repo/pilot


In [3]:
# Model load cell. Identical to notebook 12 -- native transformers
# integration, no trust_remote_code, same OOM/4-bit fallback pattern.
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor

MODEL_ID = "OpenGVLab/InternVL3-8B-hf"
QUANTIZED = False

try:
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        cache_dir=DRIVE_MODEL_CACHE,
    )
    print(f"Loaded {MODEL_ID} in bfloat16 (full precision).")
except torch.cuda.OutOfMemoryError:
    print(f"bfloat16 load of {MODEL_ID} did not fit -- falling back to 4-bit "
          "quantization. This changes what is being measured; the saved "
          "results record QUANTIZED=True so this is never silently glossed over.")
    from transformers import BitsAndBytesConfig

    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        quantization_config=quantization_config,
        device_map="auto",
        cache_dir=DRIVE_MODEL_CACHE,
    )
    QUANTIZED = True

processor = AutoProcessor.from_pretrained(MODEL_ID, cache_dir=DRIVE_MODEL_CACHE)

if torch.cuda.is_available():
    vram_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {torch.cuda.get_device_name(0)} ({vram_gib:.1f} GiB), "
          f"quantized={QUANTIZED}")


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/781 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie model.language_model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Loaded OpenGVLab/InternVL3-8B-hf in bfloat16 (full precision).
GPU: NVIDIA A100-SXM4-40GB (39.5 GiB), quantized=False


In [4]:
# Sample cell: the SAME 300-item sample notebook 12 used, filtered to
# has_error=1 (150 items) -- the pre-registered screen population, not a
# fresh draw, so results are directly comparable to the 0.628 baseline.
import ast
import glob
import logging

import pandas as pd

import pilot.data

logging.basicConfig(level=logging.INFO)

full_sample = pilot.data.load_fermat_balanced(n=300, seed=42, target_error_frac=0.5)
sample = [item for item in full_sample if item["has_error"] == 1]
N = len(sample)
print(f"Screen population: {N} has_error=1 items")
assert N == 150, f"expected 150 has_error=1 items, got {N} -- sample draw changed"

# Look up the reference run's existing 5 baseline grading samples per item,
# needed for the k10 idea (5 new + 5 existing = 10). Glob rather than a
# hardcoded timestamp, since the exact filename depends on the run.
REFERENCE_GLOB = f"{PROJECT_DIR}/results/scaleup_n300_bal50_internvl3-8b-hf_*.csv"
reference_matches = sorted(glob.glob(REFERENCE_GLOB))
if not reference_matches:
    raise FileNotFoundError(
        f"No reference CSV found at {REFERENCE_GLOB} -- the k10 idea needs "
        "notebook 12's saved reference run. Check the Drive path/filename."
    )
reference_path = reference_matches[-1]
reference_df = pd.read_csv(reference_path)
print(f"Reference run: {reference_path} ({len(reference_df)} rows)")

reference_df["_key"] = list(zip(reference_df["orig_q"], reference_df["pert_a"]))
reference_lookup = {
    row["_key"]: ast.literal_eval(row["all_grading_samples_raw"])
    for _, row in reference_df.iterrows()
}

missing = [item for item in sample if (item["orig_q"], item["pert_a"]) not in reference_lookup]
if missing:
    raise ValueError(
        f"{len(missing)}/{N} screen items have no matching reference row -- "
        "the reference CSV does not cover this sample as expected."
    )
print(f"All {N} screen items matched against the reference run's baseline samples.")


Screen population: 150 has_error=1 items
Reference run: /content/drive/MyDrive/uncertainty-math-vlm/results/scaleup_n300_bal50_internvl3-8b-hf_20260807T205407Z.csv (300 rows)
All 150 screen items matched against the reference run's baseline samples.


In [5]:
# Adapter cell -- REQUIRED pre-flight check before cell 6 runs.
# Same message shape as notebook 12 (no system role, folded into one user
# turn), parameterized by grading-prompt variant name so the same function
# serves all three screen ideas plus the k10 idea's baseline-prompt draws.
import pilot.prompts


def build_internvl_grading_messages_variant(image, variant: str) -> list[dict]:
    user_prompt = pilot.prompts.GRADING_VARIANTS[variant]
    combined_text = f"{pilot.prompts.GRADING_SYSTEM_PROMPT}\n\n{user_prompt}"
    return [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": combined_text},
            ],
        },
    ]


# Pre-flight: confirm all three variants used this run produce non-empty
# greedy output on a real sample image before spending any sampling budget.
_test_item = sample[0]
for _variant in ("commit", "restate", "baseline"):
    _test_messages = build_internvl_grading_messages_variant(_test_item["image"], _variant)
    _test_inputs = processor.apply_chat_template(
        _test_messages, tokenize=True, return_dict=True,
        return_tensors="pt", add_generation_prompt=True,
    ).to(model.device)
    with torch.no_grad():
        _test_output = model.generate(**_test_inputs, max_new_tokens=64, do_sample=False)
    _test_trimmed = _test_output[:, _test_inputs["input_ids"].shape[1]:]
    _test_text = processor.batch_decode(_test_trimmed, skip_special_tokens=True)[0]
    print(f"[{_variant}] pre-flight OK. Sample output (greedy, 64 tokens):")
    print(_test_text)
    assert len(_test_text.strip()) > 0, f"variant {_variant!r} produced empty output"


KeyError: 'commit'

In [ ]:
# Generation cell: K=5 new samples per item for "commit" and "restate";
# K=5 new *baseline*-prompt samples per item for "k10" (combined with the
# 5 existing reference samples from cell 4 during scoring). Same
# batch-backoff-ladder harness as notebooks 10/12.
import gc
import json
import os
import time

from tqdm.auto import tqdm

K_NEW = 5
TEMP = 0.7
_BATCH_LADDER = [5, 2, 1]
_batch_state = {"index": 0}

INFRA_EXCEPTIONS = (ConnectionError, TimeoutError, torch.cuda.OutOfMemoryError, OSError)

# variant name -> prompt-variant key passed to build_internvl_grading_messages_variant
SCREEN_VARIANTS = {
    "commit": "commit",
    "restate": "restate",
    "k10": "baseline",  # new draws under the ORIGINAL prompt, to combine with existing ones
}


def _generate_batch(messages, n: int, temperature: float):
    inputs = processor.apply_chat_template(
        messages, tokenize=True, return_dict=True,
        return_tensors="pt", add_generation_prompt=True,
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs, max_new_tokens=512, do_sample=True,
            temperature=temperature, num_return_sequences=n,
        )

    trimmed = output_ids[:, inputs["input_ids"].shape[1]:]
    texts = processor.batch_decode(
        trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    del output_ids, inputs
    gc.collect()
    torch.cuda.empty_cache()
    return texts


def generate_k(messages, n: int, temperature: float):
    texts = []
    last_exc = None
    while len(texts) < n:
        want = n - len(texts)
        size = min(_BATCH_LADDER[_batch_state["index"]], want)
        for attempt in range(3):
            try:
                texts += _generate_batch(messages, size, temperature)
                last_exc = None
                break
            except torch.cuda.OutOfMemoryError:
                gc.collect()
                torch.cuda.empty_cache()
                if _batch_state["index"] + 1 < len(_BATCH_LADDER):
                    _batch_state["index"] += 1
                    print(f"  OOM at batch {size}; dropping to "
                          f"{_BATCH_LADDER[_batch_state['index']]} for the rest of the run.",
                          flush=True)
                    size = min(_BATCH_LADDER[_batch_state["index"]], n - len(texts))
                    continue
                raise
            except INFRA_EXCEPTIONS as exc:
                last_exc = exc
                gc.collect()
                torch.cuda.empty_cache()
                if attempt < 2:
                    time.sleep(5)
        if last_exc is not None:
            raise last_exc
    return texts


CHECKPOINT_DIR = "/content/drive/MyDrive/uncertainty-math-vlm/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
META_FIELDS = ("orig_q", "pert_a", "has_error", "handwriting_style", "image_quality")

screen_raw = {}
for screen_name, prompt_variant in SCREEN_VARIANTS.items():
    checkpoint_path = (f"{CHECKPOINT_DIR}/screen_internvl3_{screen_name}_n{N}_seed42_k{K_NEW}.jsonl")
    entries = []
    if os.path.exists(checkpoint_path):
        with open(checkpoint_path) as f:
            entries = [json.loads(line) for line in f if line.strip()]
        valid = []
        for idx, entry in enumerate(entries[:N]):
            item = sample[idx]
            if not all(entry["item"].get(k) == item[k] for k in META_FIELDS):
                print(f"[{screen_name}] checkpoint item {idx + 1} mismatch; resuming there.")
                break
            if len(entry.get("grading_samples_raw", [])) != K_NEW:
                break
            valid.append(entry)
        if len(valid) != len(entries):
            with open(checkpoint_path, "w") as f:
                for e in valid:
                    f.write(json.dumps(e, default=str) + "\n")
        entries = valid
        print(f"[{screen_name}] resuming from {len(entries)}/{N} completed items")

    if len(entries) < N:
        print(f"[{screen_name}] generating from item {len(entries) + 1}/{N}", flush=True)
        with tqdm(total=(N - len(entries)) * K_NEW, desc=screen_name, unit="sample") as pbar:
            for item_idx, item in enumerate(sample):
                if item_idx < len(entries):
                    continue
                _t0 = time.time()
                messages = build_internvl_grading_messages_variant(item["image"], prompt_variant)
                texts = generate_k(messages, K_NEW, TEMP)
                pbar.update(K_NEW)
                entry = {
                    "item": {k: item[k] for k in META_FIELDS},
                    "grading_samples_raw": texts,
                    "quantized": QUANTIZED,
                    "elapsed_seconds": time.time() - _t0,
                }
                entries.append(entry)
                with open(checkpoint_path, "a") as f:
                    f.write(json.dumps(entry, default=str) + "\n")
                    f.flush()
    screen_raw[screen_name] = entries
    print(f"[{screen_name}]: {len(entries)} items done")


In [ ]:
# Scoring cell: score each of the 3 variants on this project's standard
# has_error=1 AUROC (entropy vs grading_correct), then apply the
# pre-registered pass bar written in the intro markdown -- BEFORE this
# cell ran for the first time, not chosen after seeing the numbers.
import pandas as pd

import pilot.canonicalize
import pilot.entropy
import pilot.parsing
import pilot.plotting


def score_variant(screen_name):
    rows = []
    for idx, entry in enumerate(screen_raw[screen_name]):
        item = sample[idx]
        if screen_name == "k10":
            key = (item["orig_q"], item["pert_a"])
            base_samples = reference_lookup[key]
            samples = list(base_samples) + list(entry["grading_samples_raw"])
        else:
            samples = entry["grading_samples_raw"]

        parsed = [pilot.parsing.parse_grading(s) for s in samples]
        labels = [None if d is None else str(d) for d in parsed]
        entropy = pilot.entropy.cluster_entropy(labels)
        majority, _ = pilot.entropy.majority_cluster(labels)
        correct = majority in {"0", "1"} and int(majority) == int(item["has_error"])
        rows.append({
            "orig_q": item["orig_q"],
            "pert_a": item["pert_a"],
            "has_error": item["has_error"],
            "reasoning_entropy": entropy,
            "grading_correct": correct,
            "k": len(samples),
            "all_grading_samples_raw": samples,
        })
    return pd.DataFrame(rows)


PRE_REGISTERED_PASS_AUROC = 0.70
PRE_REGISTERED_MIN_MINORITY = 30


def classify_screen_result(auroc_result: dict) -> str:
    """Pure function over the pre-registered bar -- written before any
    variant's numbers were seen, so the verdict can't be moved to fit."""
    n_wrong = int((~pd.Series(auroc_result["_correct"])).sum())
    n_right = len(auroc_result["_correct"]) - n_wrong
    minority = min(n_wrong, n_right)
    if minority < PRE_REGISTERED_MIN_MINORITY:
        return "inconclusive_underpowered"
    if auroc_result["auroc"] >= PRE_REGISTERED_PASS_AUROC and auroc_result["excludes_chance"]:
        return "passes_screen"
    return "does_not_pass"


screen_results = {}
for screen_name in SCREEN_VARIANTS:
    df = score_variant(screen_name)
    r = pilot.plotting.bootstrap_auroc_ci(
        df, "reasoning_entropy", "grading_correct", n_boot=10000, seed=0
    )
    r["_correct"] = df["grading_correct"].tolist()
    verdict = classify_screen_result(r)
    screen_results[screen_name] = {"df": df, "result": r, "verdict": verdict}
    print(f"[{screen_name}] AUROC {r['auroc']:.3f} [{r['ci_low']:.3f}, {r['ci_high']:.3f}] "
          f"n_wrong={int((~df['grading_correct']).sum())}  -> {verdict}")

print()
print(f"Baseline (notebook 12 reference, same 150 items): AUROC 0.628 [0.548, 0.706]")
any_pass = any(v["verdict"] == "passes_screen" for v in screen_results.values())
if any_pass:
    passing = [name for name, v in screen_results.items() if v["verdict"] == "passes_screen"]
    print(f"PASSED: {passing} -- flag for a separate full-sample confirmation decision, "
          "not an automatic next step.")
else:
    print("No variant passed the pre-registered bar. 0.628 stands as the reported "
          "InternVL3 reasoning result. Report all three results honestly -- this is "
          "the pre-registered outcome, not a failed experiment.")


In [ ]:
# Save cell: one CSV covering all 3 variants (a "screen_variant" column
# distinguishes them), plus a small summary CSV of the verdicts. Drive
# first, then repo + push -- distinct filename, never overwrites notebook
# 12's reference results.
import os
import subprocess
from datetime import datetime, timezone
from getpass import getpass

import pandas as pd

all_rows = []
for screen_name, v in screen_results.items():
    df = v["df"].copy()
    df["screen_variant"] = screen_name
    df["auroc"] = v["result"]["auroc"]
    df["auroc_ci_low"] = v["result"]["ci_low"]
    df["auroc_ci_high"] = v["result"]["ci_high"]
    df["verdict"] = v["verdict"]
    all_rows.append(df)
full_df = pd.concat(all_rows, ignore_index=True)
full_df["model_id"] = MODEL_ID
full_df["quantized"] = QUANTIZED

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
csv_name = f"internvl3_grading_screen_n{N}_{timestamp}.csv"

drive_results = "/content/drive/MyDrive/uncertainty-math-vlm/results"
os.makedirs(drive_results, exist_ok=True)
full_df.to_csv(f"{drive_results}/{csv_name}", index=False)
print(f"Backup written to {drive_results}/{csv_name}")

os.makedirs("repo/results", exist_ok=True)
csv_path = f"repo/results/{csv_name}"
full_df.to_csv(csv_path, index=False)
print(f"Wrote {csv_path} ({len(full_df)} rows)")

_REDACT = []


def git(*args):
    result = subprocess.run(["git", "-C", "repo", *args], capture_output=True, text=True)
    output = (result.stdout or "") + (result.stderr or "")
    for secret in _REDACT:
        if secret:
            output = output.replace(secret, "***")
    if result.returncode != 0 and output.strip():
        print(output.strip())
    return result


git("config", "user.email", "colab-pilot@localhost")
git("config", "user.name", "Colab Pilot Run")
git("add", f"results/{csv_name}")
commit = git("commit", "-m", f"Add InternVL3 grading-screen results: {csv_name}")
if commit.returncode != 0:
    print("git commit failed (see above) -- CSV is safe on Drive.")

GH_PUSH_TOKEN = (globals().get("GH_TOKEN") or "").strip()
if not GH_PUSH_TOKEN:
    GH_PUSH_TOKEN = getpass("GitHub token (to push results), then press Enter: ").strip()
_REDACT.append(GH_PUSH_TOKEN)

if not GH_PUSH_TOKEN:
    print("No token given -- skipping push. CSV is saved on Drive and in repo/results/.")
else:
    push_url = REPO_URL.replace("https://", f"https://{GH_PUSH_TOKEN}@")
    if git("fetch", push_url, "main").returncode == 0:
        if git("rebase", "FETCH_HEAD").returncode != 0:
            git("rebase", "--abort")
            print("Rebase onto remote failed; attempting push anyway.")
    if git("push", push_url, "HEAD:main").returncode == 0:
        print("Pushed results to the repo.")
    else:
        print("Push failed (see above). The CSV is safe on Drive and in "
              "repo/results/ -- retry the push without re-running the model.")
